# 🛡️ BioCirv AI - Stable Prototype (No-Geo)

Welcome to the BioCirv AI Stable Prototype. This environment is optimized for **reliability and performance** by focusing on non-geospatial materialized views. 

### 🎯 Objectives:
1.  **Stability**: Uses standard data types to avoid PostGIS-related crashes.
2.  **Performance**: Queries pre-aggregated materialized views for faster response times.
3.  **Trinity Results**: Every query returns the **Generated Code**, **Raw Data**, and **Interactive Plots**.

## 🚀 1. Environment Initialization

This cell handles dependency installation, authentication, and setting up the secure database tunnel.

In [ ]:
# @title Setup & Auth
import os
import sys
import subprocess
import time
import socket

def is_colab():
    return 'google.colab' in sys.modules

if is_colab():
    print("🔐 Authenticating with Google Cloud...")
    from google.colab import auth
    auth.authenticate_user()
    
    # Configuration
    repo_path = '/content/biocirv-ai'
    instance_connection_name = 'biocirv-470318:us-west1:biocirv-staging'
    os.environ['CLOUD_MODE'] = 'true'
    os.environ['DB_PORT'] = '5434'
    os.environ['DB_USER'] = 'biocirv_readonly'

    # Clone Repository (Fresh pull from dev branch)
    if os.path.exists(repo_path):
        !rm -rf {repo_path}
    print("🌐 Cloning biocirv-ai (dev branch)...")
    !git clone -b dev https://github.com/petercarbsmith/biocirv-ai.git -q
    
    # Install Dependencies
    print("📦 Installing dependencies...")
    !pip install --pre -e {repo_path} pg8000 cloud-sql-python-connector google-cloud-secret-manager pandasai -q
    
    # Setup Cloud SQL Proxy
    print("🌐 Starting Cloud SQL Proxy tunnel...")
    !pkill -9 -f cloud_sql_proxy || true
    !curl -L -o cloud_sql_proxy https://storage.googleapis.com/cloud-sql-connectors/cloud-sql-proxy/v2.14.2/cloud-sql-proxy.linux.amd64 -s
    !chmod +x cloud_sql_proxy
    
    proxy_process = subprocess.Popen(
        ['./cloud_sql_proxy', '--auto-iam-authn', '--port', '5434', instance_connection_name],
        stdout=subprocess.PIPE, stderr=subprocess.PIPE, text=True
    )
    
    # Wait for proxy
    timeout = 30
    start_time = time.time()
    while time.time() - start_time < timeout:
        try:
            with socket.create_connection(('127.0.0.1', 5434), timeout=1):
                print("✅ Cloud SQL Proxy is ready!")
                break
        except:
            time.sleep(1)
    
    # Secure Secret Retrieval
    print("🔐 Retrieving secrets...")
    from google.cloud import secretmanager
    client = secretmanager.SecretManagerServiceClient()
    
    # CBORG KEY
    name = "projects/biocirv-470318/secrets/CBORG_API_KEY/versions/latest"
    os.environ['CBORG_API_KEY'] = client.access_secret_version(request={"name": name}).payload.data.decode("UTF-8")
    
    # DB PASS
    pw_name = "projects/biocirv-470318/secrets/biocirv-staging-ro-biocirv_readonly/versions/latest"
    os.environ['DB_PASS'] = client.access_secret_version(request={"name": pw_name}).payload.data.decode("UTF-8")
    
    # Adjust sys.path for local module imports
    sys.path.append(os.path.join(repo_path, "src"))
    
    print("✅ Environment Ready!")
else:
    print("💻 Local environment detected. Ensure your .env file is configured.")

## 🤖 2. Initialize Stable AI Agent

We use the `sandbox_setup_no_geo` factory to create an agent optimized for standard data views.

In [ ]:
from ca_biositing.ai_exploration.sandbox_setup_no_geo import init_sandbox, get_agent_no_geo

llm, db_config = init_sandbox(cloud_mode=True)
agent = get_agent_no_geo(llm, db_config)

print("✅ Stable Agent Initialized!")

## 🔍 3. Connectivity Smoke Test

Verify raw database connectivity and inspect the schema metadata before proceeding.

In [ ]:
import psycopg2
import pandas as pd

print("Testing raw database connection...")
try:
    conn = psycopg2.connect(
        host='127.0.0.1',
        port='5434',
        user='biocirv_readonly',
        password=os.environ.get('DB_PASS'),
        dbname='biocirv-staging'
    )
    print("✅ Success! Raw connection established.")
    
    # Inspect views
    for view in ["ca_biositing.analysis_data_view", "ca_biositing.analysis_average_view"]:
        df = pd.read_sql(f"SELECT * FROM {view} LIMIT 5", conn)
        print(f"\n📊 Preview for {view}:")
        display(df.head())
    
    conn.close()
except Exception as e:
    print(f"❌ Connection Failed: {e}")

## 🧪 4. Sample AI Analysis

Ask a natural language question. The agent will generate SQL, execute it, and return a result containing the code, data, and answer.

In [ ]:
query = "What is the most frequent resource in the analysis_data_view?"
print(f"🤔 Query: {query}")

result = agent.chat(query)
result.display()